# FLOOD MAP v9

## Новое в v9
- **GPS-калибровка КЗС**: 3 RTK-точки у дамбы (±0.02 м)
- **Калибровка по дорогам**: OSM `ele`-теги узлов → дополнительные ground-точки
- **Маска КЗС/дамб**: дамбы никогда не затоплены (OSM + жёсткие полигоны)
- **Интерактивный слайдер**: Folium + JS, уровни 1.0–5.0 м (шаг 0.5)
- Исправлено дублирование main() (v8 bug)


In [ ]:
!pip install -q rioxarray geopandas osmnx folium rasterio mapclassify scipy
!pip install -q earthaccess h5py


In [ ]:
# ======================================================================================
# FLOOD MAP v9 — КОНФИГУРАЦИЯ
# ======================================================================================
import os, gc, warnings, json
import numpy as np
import geopandas as gpd
import pandas as pd
import rioxarray
import rasterio
import folium
from rasterio.features import shapes, rasterize as rio_rasterize
from shapely.geometry import box, Point, mapping
from scipy import ndimage
from osgeo import gdal

warnings.filterwarnings("ignore")
gdal.UseExceptions()

WD = "/kaggle/working/"
os.makedirs(WD, exist_ok=True)

INPUT_DSM  = "/kaggle/input/notebooks/ivanesin/flood-map-2-0/dem_merged_reprojected.tif"
GAUGE_CSV  = "/kaggle/input/datasets/ivanesin/gi-level/.csv"
STRIP_DATE = "2022-03-09"
CRS_METRIC = "EPSG:32636"

EARTHDATA_TOKEN = os.environ.get("EARTHDATA_TOKEN", "")

FLOOD_LEVEL = 3.05   # м БС-77

# Слайдер: уровни 1.0..5.0 шаг 0.5
FLOOD_LEVELS_SLIDER = [round(x * 0.5, 1) for x in range(2, 11)]

# EGM2008 аппроксимация
EGM_A0, EGM_A1, EGM_A2 = 16.095, 0.080, 0.010
DELTA_BS77       = +0.49
DELTA_BS77_SIGMA = 0.06
SPB_OFFSET_MIN      = 13.5
SPB_OFFSET_MAX      = 18.5
SPB_OFFSET_FALLBACK = 16.10

SMRF_SLOPE      = 0.06
SMRF_WINDOW_MAX = 70
SMRF_CELL       = 10

# GPS-точки нивелирования у КЗС (RTK, привязка к БС-77)
GPS_KZS_POINTS = [
    ("КЗС GPS-01",  59.994314, 29.694830, 4.347, "gps"),
    ("КЗС GPS-02",  59.987700, 29.696649, 3.367, "gps"),
    ("КЗС GPS-03",  60.027940, 29.836873, 3.770, "gps"),
]

CONTROL_POINTS = [
    ("ГИ (на воде)",         59.9284, 30.2701, None,  "water"),
    ("С1 ФЗ (на воде)",      59.9903, 29.6917, None,  "water"),
    ("С2 ФЗ (на воде)",      59.9903, 29.6917, None,  "water"),
    ("Ржевка ВПП (ULSS)",    59.9800, 30.5856, 17.07, "ground"),
    ("Левашово ВПП (ULLV)",  60.0867, 30.1933, 17.98, "ground"),
    ("КЗС метеоплощадка",    59.9878, 29.6967,  3.37, "ground"),
    *GPS_KZS_POINTS,
]

OSM_WATER_TAGS_LIST = [
    {'natural': 'water'}, {'landuse': 'reservoir'}, {'landuse': 'basin'},
    {'waterway': 'canal'}, {'waterway': 'river'}, {'waterway': 'dock'},
    {'waterway': 'riverbank'}, {'water': True},
]

# Жёсткие полигоны исключения дамб (WGS84 lon,lat)
DAM_EXCLUSION_POLYS_WGS84 = [
    # KZS south span (mainland to Kotlin): narrow ~400m buffer along dam
    [(29.610, 59.968), (29.610, 59.980), (29.740, 59.984), (29.740, 59.972)],
    # KZS north span (Kotlin to mainland): narrow ~400m buffer
    [(29.800, 59.996), (29.800, 60.008), (29.875, 60.012), (29.875, 60.000)],
    # Kotlin island NOT included - it is inhabited city, not dam body
]

WATER_SEEDS = [
    (30.235, 59.952), (30.275, 59.930), (30.260, 59.940),
    (30.200, 59.970), (30.130, 59.990), (30.050, 59.970),
    (29.950, 59.960), (29.850, 59.930), (29.750, 59.960),
    (30.327, 59.934), (30.365, 59.948), (30.435, 59.947),
    (30.470, 59.948), (30.500, 59.955), (30.540, 59.942),
    (30.344, 59.938), (30.322, 59.934), (30.318, 59.927),
    (30.360, 59.921), (30.295, 59.951), (30.248, 59.958), (30.272, 59.965),
    (30.335, 60.052), (30.319, 60.064), (30.296, 60.080),
    (30.137, 60.010), (30.120, 60.030),
    (29.692, 59.991), (29.800, 60.005),
    # Kronstadt / Kotlin island seeds
    (29.770, 59.993), (29.748, 59.990), (29.785, 59.995),
]

HARD_POINTS = [
    ("Ржевка ВПП центр",   59.9805, 30.5842, 17.07, "ICAO/AIP"),
    ("Левашово ВПП центр", 60.0870, 30.1920, 17.98, "ICAO/AIP"),
    ("КЗС GPS-01",         59.9943, 29.6948,  4.35, "GPS RTK"),
    ("КЗС GPS-02",         59.9877, 29.6966,  3.37, "GPS RTK"),
    ("КЗС GPS-03",         60.0279, 29.8369,  3.77, "GPS RTK"),
    ("Пулково ВПП 28L",    59.7990, 30.2570, 22.87, "ICAO/AIP"),
]
TOLERANCE_GOOD = 1.0
TOLERANCE_OK   = 3.0

CONN_RES  = 10
CLOSING_M = 20
SEED_R    = 20
MIN_AREA  = 500
SIMPLIFY  = 10

print("[CONFIG] Flood Map v9")
print(f"  FLOOD_LEVEL = {FLOOD_LEVEL} м БС-77")
print(f"  Slider: {FLOOD_LEVELS_SLIDER}")
print(f"  GPS КЗС: {len(GPS_KZS_POINTS)} точек")


In [ ]:
# ======================================================================================
# МОДУЛЬ 0: ICESat-2 ATL08 — загрузка и 3D-корегистрация (Nuth & Kääb 2011)
# ======================================================================================
# Запускается один раз. Результат (вектор сдвига) передаётся в main().
# Если NASA Earthdata не настроен — модуль пропускается (shift=0).
# ======================================================================================

from scipy.optimize import curve_fit


def download_atl08(lat_min=59.5, lat_max=60.4, lon_min=28.5, lon_max=31.5,
                   date_end=STRIP_DATE, out_dir=None):
    """Загружает ATL08 гранулы через earthaccess API."""
    try:
        import earthaccess
    except ImportError:
        print("   earthaccess не установлен: !pip install earthaccess"); return []

    out_dir = out_dir or os.path.join(WD, "atl08")
    os.makedirs(out_dir, exist_ok=True)
    existing = [f for f in os.listdir(out_dir) if f.endswith(".h5")]
    if existing:
        print(f"   ATL08: {len(existing)} файлов из кэша"); return [os.path.join(out_dir, f) for f in existing]
    if not EARTHDATA_TOKEN:
        print("   ⚠ EARTHDATA_TOKEN не задан — пропускаем ATL08"); return []

    print(f"   Поиск ATL08 {lat_min}–{lat_max}N, {lon_min}–{lon_max}E...")
    earthaccess.login(strategy="environment")
    results = earthaccess.search_data(short_name="ATL08",
        bounding_box=(lon_min, lat_min, lon_max, lat_max),
        temporal=("2018-10-14", date_end), count=50)
    print(f"   Найдено гранул: {len(results)}")
    if not results: return []
    files = earthaccess.download(results, local_path=out_dir)
    print(f"   Скачано: {len(files)} файлов"); return files


def parse_atl08(h5_path, crs_out, lat_min=59.5, lat_max=60.4,
                lon_min=28.5, lon_max=31.5,
                night_only=True, max_slope=5.0, max_unc=0.3):
    """Парсит один HDF5-файл ATL08, возвращает GeoDataFrame."""
    try:
        import h5py
    except ImportError:
        return gpd.GeoDataFrame()

    records = []
    for beam in ["gt1l", "gt1r", "gt2l", "gt2r", "gt3l", "gt3r"]:
        try:
            import h5py
            with h5py.File(h5_path, 'r') as f:
                if beam not in f or 'land_segments' not in f[beam]: continue
                ls = f[beam]['land_segments']
                lat   = ls['latitude'][:]
                lon   = ls['longitude'][:]
                h_te  = ls['terrain/h_te_best_fit'][:]
                slope = ls['terrain/terrain_slope'][:]
                unc   = ls['terrain/h_te_uncertainty'][:]
                cloud = ls['cloud_flag_atm'][:]
                msw   = ls['msw_flag'][:]
                night = ls['night_flag'][:]
        except Exception:
            continue

        geo  = (lat >= lat_min) & (lat <= lat_max) & (lon >= lon_min) & (lon <= lon_max)
        qual = (cloud == 0) & (msw == 0)
        if night_only: qual &= (night == 1)
        ok = geo & qual & (np.abs(slope) <= max_slope) & (unc <= max_unc) & (h_te > -5) & (h_te < 300)
        for i in np.where(ok)[0]:
            records.append({'lat': float(lat[i]), 'lon': float(lon[i]),
                            'h_te': float(h_te[i]), 'slope': float(slope[i])})

    if not records: return gpd.GeoDataFrame()
    df  = pd.DataFrame(records)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']),
                           crs="EPSG:4326").to_crs(crs_out)
    gdf['x'] = gdf.geometry.x; gdf['y'] = gdf.geometry.y
    return gdf


def build_stable_mask(dsm_path, crs, out_path):
    """Маска открытого рельефа (не здания, не лес, не вода) для корегистрации."""
    import osmnx as ox
    if os.path.exists(out_path):
        print("   stable_mask: из кэша"); return out_path
    ox.settings.timeout = 120

    dsm10 = os.path.join(WD, "dsm_10m_stable.tif")
    if not os.path.exists(dsm10):
        gdal.Warp(dsm10, dsm_path, options=gdal.WarpOptions(
            format="GTiff", xRes=10, yRes=10, resampleAlg="bilinear",
            dstNodata=-9999, creationOptions=["COMPRESS=LZW", "BIGTIFF=YES"]))

    with rasterio.open(dsm10) as src:
        arr       = src.read(1)
        mask      = np.ones(src.shape, dtype=np.uint8)
        if src.nodata is not None: mask[arr == src.nodata] = 0
        mask[np.isnan(arr)] = 0
        transform = src.transform; profile = src.profile.copy()
        bounds    = src.bounds

    b = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                         crs=crs).to_crs("EPSG:4326").total_bounds
    bbox = (b[0], b[1], b[2], b[3])

    for tags, label in [({'building': True}, "здания"),
                        ({'natural': ['water','wetland']}, "вода"),
                        ({'landuse': ['forest','wood']}, "лес"),
                        ({'natural': 'wood'}, "лес2")]:
        try:
            feats = ox.features_from_bbox(bbox=bbox, tags=tags)
            if feats.empty: continue
            feats = feats[feats.geometry.type.isin(['Polygon','MultiPolygon'])].to_crs(crs)
            if feats.empty: continue
            excl = rio_rasterize([(g,1) for g in feats.geometry],
                                 out_shape=mask.shape, transform=transform, fill=0, dtype='uint8')
            mask[excl == 1] = 0
            print(f"   Исключено ({label}): {int(np.sum(excl)):,} px")
        except Exception: pass

    profile.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path, 'w', **profile) as dst: dst.write(mask, 1)
    del arr, mask; gc.collect()
    return out_path


def nuth_kaab(dsm_path, atl08_gdf, stable_mask_path, crs,
              max_iter=10, conv_thr=0.05):
    """
    Алгоритм Nuth & Kääb (2011): 3D корегистрация ArcticDEM по ICESat-2.

    Физические лимиты для СПб (плоский рельеф):
      |dx|,|dy| < 10 м, |dz| < 5 м
    Защиты: RMSE-регрессия → сброс; N<30 → стоп.
    """
    if atl08_gdf.empty:
        return {'dx':0., 'dy':0., 'dz':0., 'rmse_before':np.nan, 'rmse_after':np.nan, 'n':0}

    MAX_H, MAX_V, MIN_N = 10.0, 5.0, 30
    FLAT_THR = np.deg2rad(2.0)
    print(f"   Nuth&Kääb: {len(atl08_gdf):,} ATL08-точек")

    dsm10 = os.path.join(WD, "dsm_10m_nk.tif")
    if not os.path.exists(dsm10):
        gdal.Warp(dsm10, dsm_path, options=gdal.WarpOptions(
            format="GTiff", xRes=10, yRes=10, resampleAlg="bilinear",
            dstNodata=-9999, creationOptions=["COMPRESS=LZW", "BIGTIFF=YES"]))

    with rasterio.open(dsm10) as src:
        dem   = src.read(1).astype(np.float32)
        if src.nodata is not None: dem[dem == src.nodata] = np.nan
        tf    = src.transform
        rx, ry = tf.a, abs(tf.e)

    dz_dx = ndimage.sobel(np.nan_to_num(dem), axis=1) / (8*rx)
    dz_dy = ndimage.sobel(np.nan_to_num(dem), axis=0) / (8*ry)
    slope_arr  = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    aspect_arr = np.arctan2(-dz_dy, dz_dx)

    smask = np.ones(dem.shape, dtype=np.uint8)
    if stable_mask_path and os.path.exists(stable_mask_path):
        with rasterio.open(stable_mask_path) as ms:
            sm = ms.read(1)
        if sm.shape != dem.shape:
            from scipy.ndimage import zoom as ndz
            sm = ndz(sm, (dem.shape[0]/sm.shape[0], dem.shape[1]/sm.shape[1]), order=0)
            sm = sm[:dem.shape[0], :dem.shape[1]]
        smask = sm

    def samp_bil(px, py):
        c = (px - tf.c) / tf.a; r = (py - tf.f) / tf.e
        ci = np.clip(c.astype(int), 0, dem.shape[1]-2)
        ri = np.clip(r.astype(int), 0, dem.shape[0]-2)
        fc = c-ci; fr = r-ri
        v = (1-fr)*(1-fc)*dem[ri,ci] + (1-fr)*fc*dem[ri,ci+1] +              fr*(1-fc)*dem[ri+1,ci] +     fr*fc*dem[ri+1,ci+1]
        v[np.isnan(dem[ri,ci])|np.isnan(dem[ri,ci+1])|
          np.isnan(dem[ri+1,ci])|np.isnan(dem[ri+1,ci+1])] = np.nan
        return v

    def samp_nn(px, py, arr):
        c = np.clip(((px-tf.c)/tf.a).astype(int), 0, arr.shape[1]-1)
        r = np.clip(((py-tf.f)/tf.e).astype(int), 0, arr.shape[0]-1)
        return arr[r, c]

    px, py, ph = atl08_gdf['x'].values, atl08_gdf['y'].values, atl08_gdf['h_te'].values
    cdx, cdy, cdz = 0., 0., 0.
    rmse0 = prev_rmse = None
    is_flat = float(np.nanmedian(samp_nn(px, py, slope_arr))) < FLAT_THR
    if is_flat:
        print(f"   ⚠ Плоский рельеф (<2°) → только dz")
    iter_dx = iter_dy = iter_dz = 0.

    for it in range(max_iter):
        dsm_v = samp_bil(px-cdx, py-cdy)
        dh    = dsm_v - (ph + cdz)
        sl_v  = samp_nn(px-cdx, py-cdy, slope_arr)
        asp_v = samp_nn(px-cdx, py-cdy, aspect_arr)
        st_v  = samp_nn(px-cdx, py-cdy, smask)
        ok    = ~np.isnan(dsm_v) & ~np.isnan(dh) & (st_v==1) & (np.abs(dh)<50)
        n_ok  = int(np.sum(ok))
        if n_ok < MIN_N:
            print(f"   Итерация {it+1}: N={n_ok} < {MIN_N} — стоп"); break

        dh_v  = dh[ok]; sl_ok = sl_v[ok]; asp_ok = asp_v[ok]
        rmse  = float(np.sqrt(np.mean(dh_v**2)))
        if rmse0 is None: rmse0 = rmse
        if prev_rmse and rmse > prev_rmse * 1.2 and it > 0:
            print(f"   Расхождение {prev_rmse:.3f}→{rmse:.3f} — откат"); cdx-=iter_dx; cdy-=iter_dy; cdz-=iter_dz; break
        prev_rmse = rmse

        if is_flat or np.sum(sl_ok > np.deg2rad(1)) < MIN_N:
            iter_dx = iter_dy = 0.; iter_dz = float(np.median(dh_v))
        else:
            mask_sl = sl_ok > np.deg2rad(1)
            dhn = dh_v[mask_sl] / np.tan(sl_ok[mask_sl])
            def mnk(asp, a, b, c): return a*np.cos(b-asp)+c
            try:
                po, _ = curve_fit(mnk, asp_ok[mask_sl], dhn,
                                  p0=[0.5, 0., float(np.median(dhn))], maxfev=5000)
                a, b, c = po
                iter_dx = float(a)*np.sin(b); iter_dy = float(a)*np.cos(b)
                iter_dz = float(np.median(dh_v[mask_sl] - mnk(asp_ok[mask_sl], a, b, 0)))
            except RuntimeError:
                iter_dx = iter_dy = 0.; iter_dz = float(np.median(dh_v))

        # Физические лимиты
        h = np.sqrt(iter_dx**2+iter_dy**2)
        if h > MAX_H: s=MAX_H/h; iter_dx*=s; iter_dy*=s
        if abs(iter_dz) > MAX_V: iter_dz = np.sign(iter_dz)*MAX_V

        cdx+=iter_dx; cdy+=iter_dy; cdz+=iter_dz
        tot = np.sqrt(iter_dx**2+iter_dy**2+iter_dz**2)
        print(f"   Итерация {it+1:2d}: RMSE={rmse:.3f}м "
              f"Δ=({iter_dx:+.3f},{iter_dy:+.3f},{iter_dz:+.3f})м N={n_ok}")
        if tot < conv_thr:
            print(f"   Сходимость (|Δ|={tot:.4f} < {conv_thr})"); break

    # Финальный RMSE
    dsm_f = samp_bil(px-cdx, py-cdy)
    dh_f  = dsm_f - (ph+cdz)
    vf    = ~np.isnan(dh_f) & (np.abs(dh_f)<50)
    rmse1 = float(np.sqrt(np.mean(dh_f[vf]**2))) if np.any(vf) else np.nan

    # RMSE-регрессия → сброс
    if not np.isnan(rmse1) and rmse0 and rmse1 > rmse0 * 1.05:
        print(f"   ⚠ RMSE регрессия {rmse0:.3f}→{rmse1:.3f} — СБРОС dx=dy=dz=0")
        cdx = cdy = cdz = 0.; rmse1 = rmse0

    # Суммарный лимит
    if np.sqrt(cdx**2+cdy**2) > MAX_H or abs(cdz) > MAX_V:
        print(f"   ⚠ Суммарный сдвиг нефизичен — СБРОС")
        cdx = cdy = cdz = 0.; rmse1 = rmse0

    print(f"\n   ══ Nuth&Kääb ══")
    print(f"   dx={cdx:+.3f} м  dy={cdy:+.3f} м  dz={cdz:+.3f} м")
    print(f"   RMSE: {rmse0:.3f} → {rmse1:.3f} м")
    del dem, slope_arr, aspect_arr; gc.collect()
    return {'dx':float(cdx),'dy':float(cdy),'dz':float(cdz),
            'rmse_before':float(rmse0) if rmse0 else np.nan,'rmse_after':rmse1,'n':n_ok}


def apply_shift(dsm_path, shift, out_path):
    """Применяет 3D-сдвиг к DSM (сдвиг geotransform + прибавление dz)."""
    dx, dy, dz = shift['dx'], shift['dy'], shift['dz']
    if abs(dx)<0.01 and abs(dy)<0.01 and abs(dz)<0.01:
        import shutil; shutil.copy2(dsm_path, out_path)
        print("   Сдвиг < 0.01 м — копируем без изменений"); return out_path

    src = gdal.Open(dsm_path)
    gt  = list(src.GetGeoTransform())
    gt[0] -= dx; gt[3] -= dy
    dst = gdal.GetDriverByName('GTiff').CreateCopy(
        out_path, src, options=['COMPRESS=LZW','TILED=YES','BIGTIFF=YES'])
    dst.SetGeoTransform(gt)
    if abs(dz) > 0.001:
        band = dst.GetRasterBand(1)
        nd = band.GetNoDataValue()
        for ry_ in range(0, dst.RasterYSize, 4096):
            rsz = min(4096, dst.RasterYSize-ry_)
            for rx_ in range(0, dst.RasterXSize, 4096):
                csz = min(4096, dst.RasterXSize-rx_)
                tile = band.ReadAsArray(rx_,ry_,csz,rsz).astype(np.float32)
                v = ~np.isnan(tile)
                if nd is not None: v &= (tile != nd)
                tile[v] += dz
                band.WriteArray(tile, rx_, ry_)
    dst.FlushCache(); dst = None; src = None
    print(f"   Скорректированный DSM: {out_path}")
    return out_path


print("[MODULE 0] ICESat-2 + Nuth&Kääb — загружен")


In [ ]:
# ======================================================================================
# МОДУЛИ 1–6: ВСЕ ФУНКЦИИ В ОДНОЙ ЯЧЕЙКЕ (избегаем NameError между ячейками)
# ======================================================================================
# Включает: калибровку, SMRF, water mask, threshold, connectivity, probability map
# ======================================================================================

from scipy.interpolate import griddata as sci_griddata

# ──────────────────────────────────────────────────────────────────────────────
# 1. ВЕРТИКАЛЬНАЯ КАЛИБРОВКА
# ──────────────────────────────────────────────────────────────────────────────

def egm2008_n(lat, lon):
    """EGM2008 ундуляция — полиномиальная аппроксимация, ±0.03 м для СПб."""
    N = EGM_A0 + EGM_A1*(lat-60.0) + EGM_A2*(lon-30.0)
    return float(N) if 15.0 < N < 17.5 else 16.10


def read_dsm_at(dem_path, lat, lon, crs, r=10):
    """Медиана DSM в окне r пикселей вокруг точки."""
    pt = gpd.GeoDataFrame({'geometry':[Point(lon,lat)]},crs="EPSG:4326").to_crs(crs)
    px, py = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
    ds = gdal.Open(dem_path)
    gt = ds.GetGeoTransform()
    c, row = int((px-gt[0])/gt[1]), int((py-gt[3])/gt[5])
    x0,y0 = max(0,c-r), max(0,row-r)
    xs = min(2*r, ds.RasterXSize-x0); ys = min(2*r, ds.RasterYSize-y0)
    if xs<=0 or ys<=0: ds=None; return np.nan
    arr = ds.GetRasterBand(1).ReadAsArray(x0,y0,xs,ys).astype(float)
    nd  = ds.GetRasterBand(1).GetNoDataValue(); ds = None
    if nd is not None: arr[arr==nd] = np.nan
    v = arr[~np.isnan(arr)]; v = v[(v>-5)&(v<300)]
    return float(np.median(v)) if len(v) else np.nan


def get_gauge_levels(csv_path, date_str, h0=15, h1=16):
    """Уровни водомерных постов в момент съёмки (см → м)."""
    out = {}
    try:
        df = pd.read_csv(csv_path)
        df['_dt'] = pd.to_datetime(df.iloc[:,0])
        day = df[df['_dt'].dt.strftime('%Y-%m-%d')==date_str]
        win = day[(day['_dt'].dt.hour>=h0)&(day['_dt'].dt.hour<=h1)]
        for col in ['ГИ','С1 ФЗ','С2 ФЗ']:
            if col in win.columns:
                v = pd.to_numeric(win[col],errors='coerce').dropna()/100
                if not v.empty: out[col] = round(float(v.mean()),2)
    except Exception as e: print(f"   ⚠ CSV: {e}")
    return out


def leaf_status(date_str):
    """Фенологический статус (leaf-on/off) по дате съёмки."""
    from datetime import datetime
    m = datetime.strptime(date_str,"%Y-%m-%d").month
    return ("leaf_on", 1.3) if 5<=m<=9 else ("leaf_off", 1.0)


def calibrate(dem_path, ctrl_pts, csv_path, crs, n_boot=500,
              road_elev_pts=None):
    """
    Вычисляет эмпирический total_offset и Bootstrap 95% CI.
    H_BS77 = DSM_WGS84 - offset
    Дополнительные типы: "gps" (вес=2), "road" (вес=0.5)
    """
    ctrl_pts = list(ctrl_pts) + (road_elev_pts or [])
    print(f"   {'Точка':<28} {'DSM':>7} {'N_EGM':>7} {'H_EGM':>7} {'H_BS77':>7} {'Offset':>8} Тип")
    print(f"   {'-'*83}")

    wl = get_gauge_levels(csv_path, STRIP_DATE)
    if wl: print("   Уровни: " + ", ".join(f"{k}={v:.2f}" for k,v in wl.items()))
    else:   print("   ⚠ CSV не загружен")

    lstat, csf_mult = leaf_status(STRIP_DATE)
    print(f"   Фенология: {lstat} (CSF ×{csf_mult:.1f})")

    col_map = {"ГИ (на воде)":"ГИ", "С1 ФЗ (на воде)":"С1 ФЗ", "С2 ФЗ (на воде)":"С2 ФЗ"}
    water_off, gps_off, ground_off, primary_off = [], [], [], None
    meta = []

    for name, lat, lon, h_known, ptype in ctrl_pts:
        dsm_v = read_dsm_at(dem_path, lat, lon, crs)
        n_egm = egm2008_n(lat, lon)
        if np.isnan(dsm_v): print(f"   {name:<28} N/A"); continue
        h_egm = dsm_v - n_egm

        if ptype == "water":
            h_bs77 = wl.get(col_map.get(name))
            if h_bs77 is None: print(f"   {name:<28} нет CSV"); continue
        elif ptype in ('gps', 'ground', 'road'):
            h_bs77 = h_known
        else:
            h_bs77 = h_known

        off_i = dsm_v - h_bs77
        ok = "✓" if SPB_OFFSET_MIN <= off_i <= SPB_OFFSET_MAX else "⚠"
        print(f"   {name:<28} {dsm_v:>7.2f} {n_egm:>7.2f} {h_egm:>7.2f} {h_bs77:>7.2f} {off_i:>+8.2f} {ptype} {ok}")

        if not (SPB_OFFSET_MIN <= off_i <= SPB_OFFSET_MAX):
            print(f"   {'':28} ⚠ ИСКЛЮЧЕНО (вне [{SPB_OFFSET_MIN},{SPB_OFFSET_MAX}])"); continue

        w_map = {'water': 3, 'gps': 2, 'ground': 1, 'road': 0.5}
        meta.append({'off': off_i, 'type': ptype, 'w': w_map.get(ptype,1)})
        if ptype == "water":
            water_off.append(off_i)
            if primary_off is None: primary_off = off_i
        elif ptype == 'gps':
            gps_off.append(off_i)
        else:
            ground_off.append(off_i)

    # Выбор offset: вода > GPS > наземные > road
    if primary_off is not None:
        if gps_off:
            gps_med = float(np.median(gps_off))
            delta_gps = abs(primary_off - gps_med)
            print(f"   GPS КЗС медиана: {gps_med:+.4f} м | Δ от ГИ: {delta_gps:.3f} м")
            if delta_gps < 0.3:
                all_wg = [primary_off]*3 + gps_off
                offset = float(np.mean(all_wg))
                note = f"ГИ + {len(gps_off)} GPS-КЗС (Δ={delta_gps:.3f}м)"
            else:
                offset = primary_off
                note = f"ГИ (GPS КЗС Д={delta_gps:.3f}м>нормы)"
        else:
            offset, note = primary_off, "первичный пост ГИ"
    elif gps_off:
        offset, note = float(np.median(gps_off)), f"медиана GPS ({len(gps_off)})"
    elif water_off:
        offset, note = float(np.median(water_off)), f"медиана водных ({len(water_off)})"
    elif ground_off:
        offset, note = float(np.median(ground_off)), f"медиана ВПП ({len(ground_off)})"
    else:
        offset, note = SPB_OFFSET_FALLBACK, "константа fallback"
    if ground_off and gps_off:
        print(f"   Road пт: {len(ground_off)}, GPS пт: {len(gps_off)}")

    # Bootstrap 95% CI
    vals = [m['off'] for m in meta]
    ci_lo = ci_hi = offset
    if len(vals) >= 3:
        rng = np.random.default_rng(42)
        boot = [np.median(rng.choice(vals, len(vals), replace=True)) for _ in range(n_boot)]
        ci_lo, ci_hi = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
        print(f"   Bootstrap 95% CI: [{ci_lo:.3f}, {ci_hi:.3f}] м")

    if len(water_off) > 1:
        sp = max(water_off)-min(water_off)
        print(f"   Разброс водных: {sp:.3f} м {'⚠' if sp>0.5 else ''}")

    print(f"\n   Принятый offset: {offset:+.4f} м [{note}]")
    print(f"   Порог затопления: {FLOOD_LEVEL}+{offset:.4f} = {FLOOD_LEVEL+offset:.4f} м (DSM)")
    theo = EGM_A0 + DELTA_BS77
    print(f"   Теоретический (N+δ): {EGM_A0:.2f}+{DELTA_BS77:.2f} = {theo:.2f} м  |  Δ={offset-theo:+.3f} м")
    return {'offset': offset, 'ci_lo': ci_lo, 'ci_hi': ci_hi, 'note': note,
            'csf_mult': csf_mult, 'meta': meta}


# ──────────────────────────────────────────────────────────────────────────────
# 2. МАСКА ПОКРЫТИЯ
# ──────────────────────────────────────────────────────────────────────────────

def create_coverage_mask(dsm_path, out_path, crs, res=10):
    """Маска DEM-покрытия: 1 = есть данные, 0 = nodata/за пределами."""
    dsm10 = os.path.join(WD, "dsm_10m.tif")
    if not os.path.exists(dsm10):
        gdal.Warp(dsm10, dsm_path, options=gdal.WarpOptions(
            format="GTiff", xRes=res, yRes=res, resampleAlg="bilinear",
            dstNodata=-9999, creationOptions=["COMPRESS=LZW","TILED=YES","BIGTIFF=YES"]))

    with rasterio.open(dsm10) as src:
        arr = src.read(1); nd = src.nodata; profile = src.profile.copy()
        mask = np.ones_like(arr, dtype=np.uint8)
        if nd is not None: mask[arr==nd] = 0
        mask[np.isnan(arr)] = 0

    struct = ndimage.generate_binary_structure(2,2)
    mask   = ndimage.binary_closing(mask, struct, iterations=5).astype(np.uint8)

    try:
        import osmnx as ox; ox.settings.timeout = 60
        city = ox.geocode_to_gdf("Saint Petersburg, Russia").to_crs(crs)
        with rasterio.open(dsm10) as s2:
            cm = rio_rasterize([(g,1) for g in city.geometry],
                out_shape=(s2.height,s2.width), transform=s2.transform, fill=0, dtype='uint8')
        mask &= cm; print("   + клиппинг по городу")
    except Exception as e: print(f"   ⚠ Граница: {e}")

    profile.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path,'w',**profile) as dst: dst.write(mask,1)
    print(f"   Валидных: {np.sum(mask):,} px ({100*np.sum(mask)/mask.size:.1f}%)")
    del arr, mask; gc.collect()


# ──────────────────────────────────────────────────────────────────────────────
# 3. SMRF (Simple Morphological Filter) — DSM → DTM
# ──────────────────────────────────────────────────────────────────────────────

def apply_smrf(dsm_path, dtm_out_path, crs,
               slope_thr=0.06, window_max_m=50, cell_m=10,
               cov_mask_path=None, csf_mult=1.0):
    """
    SMRF (Pingel et al. 2013, адаптация Liu et al. 2023 для Хельсинки/СПб).

    Алгоритм:
      1. Морфологическое открытие (grey_opening) с окном w = 1..window_max_px
      2. diff = surface - opened > slope_thr * w * cell_m → объект
      3. Объектные пиксели = opened (земля под ними)
      4. Итерируем по w → прогрессивно убираем здания → остаётся DTM

    Параметры СПб: slope=0.06 (6%), window=50 м (крупнопанельные дома).
    csf_mult: сезонная поправка (leaf_on ×1.3 → строже убираем листву).
    RAM: ~280 МБ для DSM СПб на 10 м.
    """
    dsm10 = os.path.join(WD, "dsm_10m.tif")
    if not os.path.exists(dsm10):
        gdal.Warp(dsm10, dsm_path, options=gdal.WarpOptions(
            format="GTiff", xRes=cell_m, yRes=cell_m, resampleAlg="bilinear",
            dstNodata=-9999, creationOptions=["COMPRESS=LZW","TILED=YES","BIGTIFF=YES"]))

    with rasterio.open(dsm10) as src:
        dsm = src.read(1).astype(np.float32)
        if src.nodata is not None: dsm[dsm==src.nodata] = np.nan
        profile = src.profile.copy()

    print(f"   DSM: {dsm.shape}, ~{dsm.nbytes/1e6:.0f} МБ")
    print(f"   SMRF: slope={slope_thr*csf_mult:.3f} (×{csf_mult:.1f}), window=1–{window_max_m} м")

    nan_mask = np.isnan(dsm)
    if np.any(nan_mask):
        from scipy.ndimage import distance_transform_edt
        idx = distance_transform_edt(nan_mask, return_distances=False, return_indices=True)
        dsm_filled = dsm[tuple(idx)]
    else:
        dsm_filled = dsm.copy()

    window_max_px = max(3, int(round(window_max_m / cell_m)))
    obj_mask = np.zeros(dsm.shape, dtype=bool)
    surface  = dsm_filled.copy()
    eff_slope = slope_thr * csf_mult   # сезонная коррекция

    for w in range(1, window_max_px+1):
        fp     = np.ones((2*w+1, 2*w+1))
        opened = ndimage.grey_opening(surface, footprint=fp)
        diff   = surface - opened
        thr_w  = eff_slope * w * cell_m
        new_obj = diff > thr_w
        obj_mask |= new_obj
        surface[new_obj] = opened[new_obj]
        if w % max(1, window_max_px//5) == 0 or w == window_max_px:
            pct = 100*np.sum(obj_mask&~nan_mask)/max(1,np.sum(~nan_mask))
            print(f"     w={w:2d} ({w*cell_m:3d} м): объектов {pct:.1f}%")

    if cov_mask_path and os.path.exists(cov_mask_path):
        with rasterio.open(cov_mask_path) as cs:
            cov = cs.read(1)
        if cov.shape != dsm.shape:
            from scipy.ndimage import zoom as ndz
            cov = ndz(cov,(dsm.shape[0]/cov.shape[0],dsm.shape[1]/cov.shape[1]),order=0)
            cov = cov[:dsm.shape[0],:dsm.shape[1]]
        obj_mask[cov==0] = False

    n_obj  = int(np.sum(obj_mask&~nan_mask))
    n_gnd  = int(np.sum(~obj_mask&~nan_mask))
    print(f"   Объектов: {n_obj:,} ({100*n_obj/max(1,n_obj+n_gnd):.1f}%) | Земля: {n_gnd:,}")

    dtm = surface.copy()
    dtm = ndimage.uniform_filter(dtm, size=3)
    valid = ~nan_mask
    dtm[valid] = np.minimum(dtm[valid], dsm[valid])
    dtm[nan_mask] = -9999

    v_vals = dtm[valid]
    if len(v_vals):
        print(f"   DTM: мин={v_vals.min():.2f}, макс={v_vals.max():.2f}, медиана={np.median(v_vals):.2f} м")

    if obj_mask.any() and (obj_mask&valid).any():
        ndsm = dsm - dtm
        obj_v = ndsm[valid&obj_mask]
        print(f"   nDSM (объекты): медиана={np.median(obj_v):.1f} м, P95={np.percentile(obj_v,95):.1f} м")

    profile.update(dtype='float32', count=1, nodata=-9999)
    with rasterio.open(dtm_out_path,'w',**profile) as dst:
        dst.write(dtm.astype(np.float32),1)
    del dsm, dsm_filled, surface, dtm, obj_mask; gc.collect()
    print(f"   DTM (SMRF): {dtm_out_path}")
    return dtm_out_path


# ──────────────────────────────────────────────────────────────────────────────
# 4. OSM WATER MASK — все типы водных объектов
# ──────────────────────────────────────────────────────────────────────────────

def build_osm_water_mask(dsm_path, out_path, crs, res=10):
    """
    Растровая маска всех OSM-водоёмов (10 м, uint8).

    Использует OSM_WATER_TAGS_LIST из конфига (8 тегов).
    Критично для каналов СПб: waterway=canal захватывает
    Фонтанку, Мойку, Грибоедова, Обводный, Зимнюю канавку.
    """
    import osmnx as ox
    ox.settings.timeout = 120

    if os.path.exists(out_path):
        with rasterio.open(out_path) as s:
            n = int(np.sum(s.read(1)))
        print(f"   OSM water mask: из кэша ({n:,} px)"); return out_path

    ref = os.path.join(WD, "dsm_10m.tif")
    if not os.path.exists(ref):
        gdal.Warp(ref, dsm_path, options=gdal.WarpOptions(
            format="GTiff", xRes=res, yRes=res, resampleAlg="bilinear",
            dstNodata=-9999, creationOptions=["COMPRESS=LZW","BIGTIFF=YES"]))

    with rasterio.open(ref) as src:
        shape=src.shape; tf=src.transform; prof=src.profile.copy(); bounds=src.bounds

    b = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                         crs=crs).to_crs("EPSG:4326").total_bounds
    bbox = (b[0],b[1],b[2],b[3])

    parts = []
    tag_counts = {}
    for tags in OSM_WATER_TAGS_LIST:
        try:
            raw = ox.features_from_bbox(bbox=bbox, tags=tags)
            if raw.empty: continue
            poly = raw[raw.geometry.type.isin(['Polygon','MultiPolygon'])][['geometry']].to_crs(crs)
            if not poly.empty:
                parts.append(poly)
                tag_counts[str(tags)] = len(poly)
        except Exception: pass

    if parts:
        combined = gpd.GeoDataFrame(
            pd.concat(parts,ignore_index=True), crs=crs).drop_duplicates(subset='geometry')
        mask = rio_rasterize([(g,1) for g in combined.geometry],
            out_shape=shape, transform=tf, fill=0, dtype='uint8')
        print(f"   OSM water mask: {len(combined):,} полигонов → {int(np.sum(mask)):,} px (10 м)")
        for t,n in tag_counts.items(): print(f"     {t}: {n}")
    else:
        mask = np.zeros(shape, dtype=np.uint8)
        print("   ⚠ OSM water mask: нет данных")

    prof.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path,'w',**prof) as dst: dst.write(mask,1)
    del mask; gc.collect()
    return out_path


# ──────────────────────────────────────────────────────────────────────────────
# 4b. WATER FLATTENING (тайловый, in-place на 10 м DSM)
# ──────────────────────────────────────────────────────────────────────────────

def water_flattening(crs, offset, gauge_wl=None, tile_sz=2048):
    """
    Выравнивает водные поверхности в dsm_10m.tif (in-place, тайловый).
    OOM-защита: загружает только маску (uint8) + тайл DSM.
    """
    import osmnx as ox
    ox.settings.timeout = 120
    dsm10 = os.path.join(WD, "dsm_10m.tif")
    if not os.path.exists(dsm10): print("   ⚠ dsm_10m.tif не найден"); return

    with rasterio.open(dsm10) as src:
        rows,cols  = src.shape; tf = src.transform; nd_val = src.nodata; bounds = src.bounds

    b = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                         crs=crs).to_crs("EPSG:4326").total_bounds
    bbox = (b[0],b[1],b[2],b[3])

    # Загружаем уже готовую маску (если есть)
    wm_path = os.path.join(WD, "osm_water_mask.tif")
    if os.path.exists(wm_path):
        with rasterio.open(wm_path) as ws: wm = ws.read(1)
        if wm.shape != (rows,cols):
            from scipy.ndimage import zoom as ndz
            wm = ndz(wm,(rows/wm.shape[0],cols/wm.shape[1]),order=0).astype(np.uint8)
            wm = wm[:rows,:cols]
    else:
        wm = np.zeros((rows,cols), dtype=np.uint8)

    wl_bs77   = gauge_wl.get('ГИ', 0.0) if gauge_wl else 0.0
    sea_dsm   = offset + wl_bs77

    src_ds = gdal.Open(dsm10, gdal.GA_Update); band = src_ds.GetRasterBand(1)
    n_flat = 0
    nty = (rows+tile_sz-1)//tile_sz
    for ty in range(nty):
        yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
        for tx in range((cols+tile_sz-1)//tile_sz):
            xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
            wt = wm[yo:yo+ys, xo:xo+xs]
            if np.sum(wt)==0: continue
            tile = band.ReadAsArray(xo,yo,xs,ys).astype(np.float32)
            vw = (wt==1)
            if nd_val is not None: vw &= (tile!=nd_val)
            tile[vw] = sea_dsm
            band.WriteArray(tile,xo,yo)
            n_flat += int(np.sum(vw))
    src_ds.FlushCache(); src_ds = None
    print(f"   Water flattening: {n_flat:,} px → {sea_dsm:.3f} м WGS84")

    # Нева: градиент 0 (запад) → 5.1 м (восток) БС-77
    NEVA_X_WEST = 304_500.0; NEVA_X_EAST = 361_200.0  # UTM36N fallback
    try:
        neva = ox.features_from_bbox(bbox=bbox, tags={'waterway': 'river', 'name': 'Нева'})
        if not neva.empty and not neva.geometry.isnull().all():
            neva = neva.to_crs(crs)
            tb = neva.total_bounds; xmn, xmx = tb[0], tb[2]
            if xmx - xmn < 40_000:
                print(f"   Нева OSM: {(xmx-xmn)/1000:.1f} км < 40 км → fallback")
                xmn, xmx = NEVA_X_WEST, NEVA_X_EAST
            nbuf = neva.geometry.buffer(150)
        else:
            print("   Нева OSM пуст → fallback")
            xmn, xmx = NEVA_X_WEST, NEVA_X_EAST
            from shapely.geometry import box as _sb
            nbuf = gpd.GeoDataFrame(
                {'geometry': [_sb(xmn, 6_626_000, xmx, 6_641_000)]}, crs=crs).geometry.buffer(0)
        if xmx > xmn:
            nm = rio_rasterize([(g,1) for g in nbuf],
                out_shape=(rows,cols), transform=tf, fill=0, dtype='uint8')
            src2 = gdal.Open(dsm10, gdal.GA_Update); b2 = src2.GetRasterBand(1)
            for ty in range(nty):
                yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
                for tx in range((cols+tile_sz-1)//tile_sz):
                    xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
                    nt = nm[yo:yo+ys, xo:xo+xs]
                    if np.sum(nt)==0: continue
                    cc = (np.arange(xs)+xo)*tf.a + tf.c
                    Xg = np.tile(cc, (ys,1)).astype(np.float32)
                    h  = 5.1*np.clip((Xg-xmn)/(xmx-xmn), 0, 1)
                    tile = b2.ReadAsArray(xo, yo, xs, ys).astype(np.float32)
                    tile[nt==1] = (offset+h)[nt==1]
                    b2.WriteArray(tile, xo, yo)
            src2.FlushCache(); src2=None
            print(f"   Нева: градиент {(xmx-xmn)/1000:.0f} км применён")
    except Exception as e: print(f"   Нева: {e}")
    del wm; gc.collect()


# ──────────────────────────────────────────────────────────────────────────────
# 5. ПОРОГОВАЯ МАСКА 2 М + СВЯЗНОСТЬ + КАРТА ВЕРОЯТНОСТИ
# ──────────────────────────────────────────────────────────────────────────────

def threshold_mask_tiled(dsm_path, out_path, threshold, water_level,
                          water_mask_path=None, tile_sz=4096):
    """
    Тайловая пороговая маска с форсированием OSM-воды.
    water_mask_path: OSM-маска 10 м → апсэмплинг до 2 м → принудительно =1.
    """
    src    = gdal.Open(dsm_path)
    cols,rows = src.RasterXSize, src.RasterYSize
    band   = src.GetRasterBand(1); src_nd = band.GetNoDataValue()
    dst    = gdal.GetDriverByName('GTiff').Create(
        out_path, cols, rows, 1, gdal.GDT_Byte,
        ['COMPRESS=LZW','TILED=YES','BLOCKXSIZE=256','BLOCKYSIZE=256','BIGTIFF=YES'])
    dst.SetGeoTransform(src.GetGeoTransform()); dst.SetProjection(src.GetProjection())
    dst.GetRasterBand(1).SetNoDataValue(255)

    # OSM water mask (10 м → 2 м апсэмплинг блоками, не весь массив)
    # Загружаем маску целиком (uint8, 10 м → размер небольшой)
    wm_arr = None
    if water_mask_path and os.path.exists(water_mask_path):
        with rasterio.open(water_mask_path) as wm_src:
            wm_raw = wm_src.read(1)
        # Апсэмплируем к размеру 2м DSM через nearest (order=0)
        from scipy.ndimage import zoom as ndz
        zy = rows/wm_raw.shape[0]; zx = cols/wm_raw.shape[1]
        wm_arr = (ndz(wm_raw,(zy,zx),order=0)>0).astype(np.uint8)
        wm_arr = wm_arr[:rows,:cols]
        print(f"   OSM water mask: {int(np.sum(wm_arr)):,} px принудительно затоплены (2м)")
        del wm_raw

    nty = (rows+tile_sz-1)//tile_sz
    for ty in range(nty):
        yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
        for tx in range((cols+tile_sz-1)//tile_sz):
            xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
            tile = band.ReadAsArray(xo,yo,xs,ys).astype(np.float32)
            nd_m = np.isnan(tile)
            if src_nd is not None: nd_m |= (tile==src_nd)
            tile[nd_m] = water_level
            mask = (tile<=threshold).astype(np.uint8)
            if wm_arr is not None:
                wt = wm_arr[yo:yo+ys,xo:xo+xs]
                mask = np.maximum(mask,wt)
            dst.GetRasterBand(1).WriteArray(mask,xo,yo)
        if (ty+1)%max(1,nty//5)==0 or ty==nty-1: print(f"     {100*(ty+1)/nty:.0f}%")
    dst.FlushCache(); dst=None; src=None
    if wm_arr is not None: del wm_arr


def connectivity_analysis(mask_2m, out_conn, seeds, crs,
                           closing_m, conn_res, seed_r,
                           cov_path=None, dtm_path=None, threshold=None):
    """Связность на 10 м + DTM-коррекция + фильтр по coverage."""
    mask10 = os.path.join(WD, "mask_10m.tif")
    gdal.Warp(mask10, mask_2m, options=gdal.WarpOptions(
        format="GTiff",xRes=conn_res,yRes=conn_res,resampleAlg="max",
        creationOptions=["COMPRESS=LZW","TILED=YES"]))
    with rasterio.open(mask10) as src:
        arr=src.read(1); tf=src.transform; prof=src.profile.copy()
    print(f"   Маска 10 м: {arr.shape}")

    if dtm_path and os.path.exists(dtm_path) and threshold is not None:
        with rasterio.open(dtm_path) as ds: dtm=ds.read(1).astype(np.float32)
        dtm[dtm==-9999]=np.nan
        dm = (~np.isnan(dtm)&(dtm<=threshold)).astype(np.uint8)
        n  = int(np.sum((arr==0)&(dm==1)))
        arr = arr|dm; print(f"   + DTM добавил {n:,} px"); del dtm,dm

    cpx = max(1,int(round(closing_m/conn_res)))
    arr = ndimage.binary_closing(arr, ndimage.generate_binary_structure(2,2),
                                  iterations=cpx).astype(np.uint8)
    labels,nf = ndimage.label(arr); print(f"   Компонент: {nf}")

    valid = set()
    for lon,lat in seeds:
        pt=gpd.GeoDataFrame({'geometry':[Point(lon,lat)]},crs="EPSG:4326").to_crs(crs)
        sx,sy=pt.geometry.iloc[0].x,pt.geometry.iloc[0].y
        c=int((sx-tf.c)/tf.a); r=int((sy-tf.f)/tf.e)
        r0,r1=max(0,r-seed_r),min(labels.shape[0],r+seed_r+1)
        c0,c1=max(0,c-seed_r),min(labels.shape[1],c+seed_r+1)
        found=set(np.unique(labels[r0:r1,c0:c1]))-{0}
        if found: valid.update(found)
        else: print(f"   ⚠ Seed ({lon:.3f},{lat:.3f}) — нет маски в {seed_r*conn_res} м")

    if not valid:
        for lb in range(1,nf+1):
            if np.sum(labels==lb)>5000: valid.add(lb)

    conn = np.isin(labels,list(valid)).astype(np.uint8)

    if cov_path and os.path.exists(cov_path):
        with rasterio.open(cov_path) as cs: cov=cs.read(1)
        if cov.shape!=conn.shape:
            from scipy.ndimage import zoom as ndz
            cov=ndz(cov,(conn.shape[0]/cov.shape[0],conn.shape[1]/cov.shape[1]),order=0)
            cov=cov[:conn.shape[0],:conn.shape[1]]
        conn &= cov

    nf2=int(np.sum(conn))
    print(f"   Итого: {nf2:,} px (~{nf2*conn_res**2/10000:.0f} га)")
    prof.update(dtype='uint8',nodata=255)
    with rasterio.open(out_conn,'w',**prof) as dst: dst.write(conn,1)
    del labels,arr,conn; gc.collect()


def apply_filter_2m(mask_2m, conn_10m, out_final, tile_sz=4096):
    """Апсэмплинг 10м связности → AND с 2м маской."""
    conn_2m = os.path.join(WD,"conn_2m.tif")
    sm = gdal.Open(mask_2m); gt = sm.GetGeoTransform()
    gdal.Warp(conn_2m,conn_10m,options=gdal.WarpOptions(
        format="GTiff",xRes=abs(gt[1]),yRes=abs(gt[5]),resampleAlg="near",
        creationOptions=["COMPRESS=LZW","TILED=YES","BIGTIFF=YES"]))
    sf = gdal.Open(conn_2m)
    cols,rows = sm.RasterXSize,sm.RasterYSize
    dst = gdal.GetDriverByName('GTiff').Create(out_final,cols,rows,1,gdal.GDT_Byte,
          ['COMPRESS=LZW','TILED=YES','BIGTIFF=YES'])
    dst.SetGeoTransform(gt); dst.SetProjection(sm.GetProjection())
    for ty in range((rows+tile_sz-1)//tile_sz):
        yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
        for tx in range((cols+tile_sz-1)//tile_sz):
            xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
            m=sm.GetRasterBand(1).ReadAsArray(xo,yo,xs,ys)
            fc,fr=sf.RasterXSize,sf.RasterYSize
            res=np.zeros_like(m)
            if xo<fc and yo<fr:
                fxs=min(xs,fc-xo); fys=min(ys,fr-yo)
                f=sf.GetRasterBand(1).ReadAsArray(xo,yo,fxs,fys)
                res[:fys,:fxs]=m[:fys,:fxs]&f
            dst.GetRasterBand(1).WriteArray(res,xo,yo)
    dst.FlushCache(); dst=sm=sf=None
    if os.path.exists(conn_2m): os.remove(conn_2m)


# ──────────────────────────────────────────────────────────────────────────────
# 6. КАРТА ВЕРОЯТНОСТИ ЗАТОПЛЕНИЯ (Монте-Карло по CI offset)
# ──────────────────────────────────────────────────────────────────────────────

def build_probability_map(dsm_path, out_path, calib, flood_level=3.05,
                           n_samples=50, tile_sz=4096):
    """
    P(flood) в каждом пикселе с учётом неопределённости калибровки.
    offset ~ Uniform(ci_lo, ci_hi); threshold_i = flood_level + offset_i.
    Результат: uint8 растр 0–255 (0%–100% вероятности).
    """
    thresholds = flood_level + np.linspace(calib['ci_lo'], calib['ci_hi'], n_samples)
    print(f"   Prob map: {n_samples} симуляций, "
          f"CI=[{calib['ci_lo']:.3f},{calib['ci_hi']:.3f}] м")

    src  = gdal.Open(dsm_path)
    cols,rows = src.RasterXSize,src.RasterYSize
    band = src.GetRasterBand(1); src_nd = band.GetNoDataValue()
    dst  = gdal.GetDriverByName('GTiff').Create(out_path,cols,rows,1,gdal.GDT_Byte,
           ['COMPRESS=LZW','TILED=YES','BIGTIFF=YES'])
    dst.SetGeoTransform(src.GetGeoTransform()); dst.SetProjection(src.GetProjection())
    dst.GetRasterBand(1).SetNoDataValue(255)

    nty=(rows+tile_sz-1)//tile_sz
    for ty in range(nty):
        yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
        for tx in range((cols+tile_sz-1)//tile_sz):
            xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
            tile = band.ReadAsArray(xo,yo,xs,ys).astype(np.float32)
            nd_m = np.isnan(tile)
            if src_nd is not None: nd_m |= (tile==src_nd)
            cnt  = np.zeros_like(tile)
            for thr in thresholds: cnt += (tile<=thr).astype(np.float32)
            prob = (cnt/n_samples*255).astype(np.uint8)
            prob[nd_m] = 255
            dst.GetRasterBand(1).WriteArray(prob,xo,yo)
    dst.FlushCache(); dst=src=None
    print(f"   Prob map: {out_path}")


# ──────────────────────────────────────────────────────────────────────────────
# 7. ВЕКТОРИЗАЦИЯ + FOLIUM КАРТА
# ──────────────────────────────────────────────────────────────────────────────

def vectorize_and_map(final_mask, calib, shift, crs, out_html):
    """Векторизация + Folium карта с мета-панелью."""
    with rasterio.open(final_mask) as src:
        arr=src.read(1); tf=src.transform; bounds=src.bounds
    print(f"   Затопленных: {int(np.sum(arr)):,} px")
    if np.sum(arr)==0: print("   Нет затопления."); return

    geoms=[{'properties':{'v':int(v)},'geometry':s}
           for s,v in shapes(arr,mask=arr==1,transform=tf)]
    del arr; gc.collect()
    if not geoms: return

    gdf = gpd.GeoDataFrame.from_features(geoms,crs=crs); del geoms; gc.collect()
    gdf = gdf[gdf.geometry.area>=MIN_AREA].dissolve()
    gdf.geometry = gdf.geometry.simplify(SIMPLIFY)
    ha = gdf.geometry.area.sum()/10000
    print(f"   Площадь: {ha:.0f} га")

    bw = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                          crs=crs).to_crs("EPSG:4326").total_bounds
    cy,cx = (bw[1]+bw[3])/2, (bw[0]+bw[2])/2
    m = folium.Map(location=[cy,cx],zoom_start=11,tiles='CartoDB positron')
    fw = gdf.to_crs("EPSG:4326")
    folium.GeoJson(fw,name='Зона затопления',
        style_function=lambda x:{'fillColor':'#1E90FF','color':'#0000CD',
                                  'weight':0.5,'fillOpacity':0.55}).add_to(m)

    info = f"""<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
    background:white;padding:12px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,.3);
    font-size:12px;line-height:1.6em">
    <b>Flood Map v8</b><br>
    Уровень: {FLOOD_LEVEL} м БС-77<br>
    Дата стрипов: {STRIP_DATE}<br>
    Offset: {calib['offset']:.3f} м [{calib['note']}]<br>
    CI 95%: [{calib['ci_lo']:.3f}, {calib['ci_hi']:.3f}] м<br>
    Coreg dz: {shift['dz']:+.3f} м (RMSE {shift['rmse_after']:.3f}→{shift['rmse_before']:.3f})<br>
    DTM: SMRF (slope={SMRF_SLOPE}, w={SMRF_WINDOW_MAX} м)<br>
    Площадь: <b>{ha:.0f} га</b>
    </div>"""
    m.get_root().html.add_child(folium.Element(info))

    try:
        import osmnx as ox; ox.settings.timeout=60
        fb=fw.total_bounds
        lu=ox.features_from_bbox(bbox=(fb[0],fb[1],fb[2],fb[3]),tags={'landuse':True})
        if not lu.empty:
            lu=lu[lu.geometry.type.isin(['Polygon','MultiPolygon'])].to_crs(crs)
            imp=gpd.overlay(lu,gdf,how='intersection')
            imp['ha']=imp.geometry.area/10000
            print("\n   Landuse в зоне затопления:")
            print(imp.groupby('landuse')['ha'].sum().sort_values(ascending=False).head(10).to_string())
    except Exception as e: print(f"   ⚠ landuse: {e}")
    # DEM topo heatmap layer
    try:
        import matplotlib; matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import matplotlib.colors as mcolors
        import base64
        from PIL import Image as PILImage

        dem_png = os.path.join(WD, 'dem_heatmap.png')
        with rasterio.open(dtm_path) as ds:
            scale = 6
            dtm_sm = ds.read(1,
                out_shape=(ds.height//scale, ds.width//scale),
                resampling=rasterio.enums.Resampling.average).astype(np.float32)
            raw_bounds = ds.bounds; raw_crs = ds.crs
        dtm_sm[dtm_sm == -9999] = np.nan

        bnd4 = gpd.GeoDataFrame(
            {'geometry':[box(raw_bounds.left,raw_bounds.bottom,
                             raw_bounds.right,raw_bounds.top)]},
            crs=raw_crs).to_crs('EPSG:4326').total_bounds
        img_bounds = [[float(bnd4[1]),float(bnd4[0])],[float(bnd4[3]),float(bnd4[2])]]

        norm = mcolors.Normalize(vmin=-2, vmax=60)
        cmap = plt.get_cmap('terrain')
        rgba = (cmap(norm(np.where(np.isnan(dtm_sm), -10, dtm_sm))) * 255).astype(np.uint8)
        rgba[np.isnan(dtm_sm)] = [50, 100, 180, 0]
        pil = PILImage.fromarray(rgba, 'RGBA').transpose(PILImage.FLIP_TOP_BOTTOM)
        pil.save(dem_png)

        with open(dem_png,'rb') as _f: dem_b64 = base64.b64encode(_f.read()).decode()
        folium.raster_layers.ImageOverlay(
            image='data:image/png;base64,'+dem_b64,
            bounds=img_bounds, opacity=0.72,
            name='Рельеф DTM (тепловая карта)',
            show=False, interactive=False,
        ).add_to(m)

        # Colorbar as fixed div
        fig_cb, ax_cb = plt.subplots(figsize=(3.8, 0.35))
        fig_cb.subplots_adjust(bottom=0.55)
        matplotlib.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal')
        ax_cb.set_xlabel('Высота DTM, м (прибл.)', fontsize=7)
        cb_path = os.path.join(WD,'dem_colorbar.png')
        fig_cb.savefig(cb_path,dpi=72,bbox_inches='tight',transparent=True); plt.close(fig_cb)
        with open(cb_path,'rb') as _f: cb_b64 = base64.b64encode(_f.read()).decode()

        # Build elevation grid for click queries
        step = max(1, dtm_sm.shape[0]//180)
        grid = dtm_sm[::step,::step]
        gr, gc = grid.shape
        gflat = [round(float(v),1) if not np.isnan(v) else None for v in grid.flatten()]

        dem_ui = (
            '<div id="dem-cb" style="display:none;position:fixed;bottom:98px;left:10px;'
            'z-index:950;background:rgba(255,255,255,0.93);padding:7px 10px;'
            'border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.18);font-size:10.5px">'
            '<img src="data:image/png;base64,'+cb_b64+'" style="width:250px"><br>'
            '<span id="dem-info">Включите слой и кликните на карту</span></div>'
        )
        m.get_root().html.add_child(folium.Element(dem_ui))

        _gr = str(gr); _gc = str(gc)
        _ib = (str(img_bounds[0][0]) + ',' + str(img_bounds[0][1]) + ',' +
               str(img_bounds[1][0]) + ',' + str(img_bounds[1][1]))
        _gd = json.dumps(gflat)
        dem_js_code = (
            '<script>(function(){'
            'var GR=' + _gr + ',GC=' + _gc + ';'
            'var GD=' + _gd + ';'
            'var IB={s:' + str(img_bounds[0][0]) + ',w:' + str(img_bounds[0][1]) +
                    ',n:' + str(img_bounds[1][0]) + ',e:' + str(img_bounds[1][1]) + '};'
            'var vis=false;'
            'function gElev(lat,lng){'
            '  if(lat<IB.s||lat>IB.n||lng<IB.w||lng>IB.e)return null;'
            '  var r=Math.round((IB.n-lat)/(IB.n-IB.s)*(GR-1));'
            '  var c=Math.round((lng-IB.w)/(IB.e-IB.w)*(GC-1));'
            '  r=Math.max(0,Math.min(GR-1,r));c=Math.max(0,Math.min(GC-1,c));'
            '  return GD[r*GC+c];'
            '}'
            'document.addEventListener("DOMContentLoaded",function(){'
            '  setTimeout(function(){'
            '    var mp=(function(){'
            '      var e=document.querySelectorAll(".leaflet-container");'
            '      for(var i=0;i<e.length;i++){if(e[i]._leaflet_map)return e[i]._leaflet_map;}'
            '      for(var k in window){try{if(window[k]&&window[k].addLayer)return window[k];}catch(ex){}}'
            '      return null;})();'
            '    if(!mp)return;'
            '    mp.on("click",function(e){'
            '      if(!vis)return;'
            '      var el=gElev(e.latlng.lat,e.latlng.lng);'
            '      var t=(el!==null&&el!==undefined)?'
            '        e.latlng.lat.toFixed(5)+", "+e.latlng.lng.toFixed(5)+": <b>"+el+" m</b> DTM"'
            '        :"no data";'
            '      document.getElementById("dem-info").innerHTML=t;'
            '    });'
            '    mp.on("overlayadd",function(e){'
            '      if(e.name&&e.name.indexOf("\u0420\u0435\u043b\u044c\u0435\u0444")>=0){'
            '        vis=true;document.getElementById("dem-cb").style.display="block";}'
            '    });'
            '    mp.on("overlayremove",function(e){'
            '      if(e.name&&e.name.indexOf("\u0420\u0435\u043b\u044c\u0435\u0444")>=0){'
            '        vis=false;document.getElementById("dem-cb").style.display="none";}'
            '    });'
            '  },1200);});'
            '})();</script>'
        )
        m.get_root().html.add_child(folium.Element(dem_js_code))
        print("   DEM heatmap layer: OK")
    except Exception as _dem_err:
        print(f"   DEM heatmap skip: {_dem_err}")

    folium.LayerControl().add_to(m)
    m.save(out_html); print(f"   Карта: {out_html}")
    return m




# ─────────────────────────────────────────────────────────────────────────────
# v9: НОВЫЕ ФУНКЦИИ
# ─────────────────────────────────────────────────────────────────────────────

def extract_road_elevations(dsm_path, crs, max_pts=300):
    """
    Извлекает OSM-узлы дорог с тегом `ele` и проверяет их на DSM.
    Возвращает list of (name, lat, lon, h_bs77, "road").
    Используется как дополнительный источник калибровки (вес=0.5).
    """
    try:
        import osmnx as ox
        ox.settings.timeout = 60
    except ImportError:
        return []

    print("   Road ele: загрузка OSM-узлов с тегом ele...")
    try:
        ds = gdal.Open(dsm_path)
        gt = ds.GetGeoTransform()
        bb_rect = [gt[0], gt[3]+gt[5]*ds.RasterYSize, gt[0]+gt[1]*ds.RasterXSize, gt[3]]
        ds = None
        bm = gpd.GeoDataFrame({'geometry':[box(bb_rect[0],bb_rect[1],bb_rect[2],bb_rect[3])]},
                               crs=crs).to_crs("EPSG:4326").total_bounds
        bbox_wgs = (float(bm[0]), float(bm[1]), float(bm[2]), float(bm[3]))
    except Exception as e:
        print(f"   Road ele: bbox err: {e}"); return []

    try:
        G = ox.graph_from_bbox(bbox=bbox_wgs, network_type='drive', simplify=True)
        nodes, _ = ox.graph_to_gdfs(G)
    except Exception as e:
        print(f"   Road ele: graph err: {e}"); return []

    if 'ele' not in nodes.columns:
        print("   Road ele: тег ele не найден в OSM"); return []

    nodes_ele = nodes[nodes['ele'].notna()].copy()
    print(f"   Road ele: {len(nodes_ele)} узлов с ele из {len(nodes)}")
    if len(nodes_ele) == 0: return []

    def parse_ele(v):
        try: return float(str(v).replace(',','.').split(';')[0].strip())
        except: return np.nan

    nodes_ele['_ele'] = nodes_ele['ele'].apply(parse_ele)
    nodes_ele = nodes_ele[nodes_ele['_ele'].notna() & (nodes_ele['_ele'] >= 0) & (nodes_ele['_ele'] <= 200)]
    nodes_m = nodes_ele.to_crs(crs)

    pts = []
    for idx, row in nodes_m.iterrows():
        lat = float(nodes_ele.loc[idx,'y']) if 'y' in nodes_ele.columns else row.geometry.y
        lon = float(nodes_ele.loc[idx,'x']) if 'x' in nodes_ele.columns else row.geometry.x
        h_osm = float(row['_ele'])
        dsm_v = read_dsm_at(dsm_path, lat, lon, crs, r=5)
        if np.isnan(dsm_v): continue
        off_i = dsm_v - h_osm
        if not (SPB_OFFSET_MIN <= off_i <= SPB_OFFSET_MAX): continue
        pts.append((f"road_{idx}", lat, lon, h_osm, "road"))
        if len(pts) >= max_pts: break

    print(f"   Road ele: {len(pts)} валидных точек")
    return pts


def build_dam_mask(crs, out_path, res=10):
    """
    Маска дамб КЗС — пиксели, которые никогда не должны быть затоплены.
    Источники: OSM barrier=dam/dyke + E18 bridge + жёсткие полигоны из конфига.
    """
    if os.path.exists(out_path):
        print(f"   Dam mask: кэш OK"); return out_path

    import osmnx as ox
    ox.settings.timeout = 120

    ref = os.path.join(WD, "dsm_10m.tif")
    if not os.path.exists(ref):
        print("   Dam mask: dsm_10m.tif не найден"); return None

    with rasterio.open(ref) as src:
        shape = src.shape; tf = src.transform; prof = src.profile.copy(); bounds = src.bounds

    b = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                         crs=crs).to_crs("EPSG:4326").total_bounds
    bbox = (b[0],b[1],b[2],b[3])
    all_geoms = []

    for tags in [{'barrier':'dam'},{'man_made':'dyke'},{'waterway':'dam'},{'embankment':True}]:
        try:
            f = ox.features_from_bbox(bbox=bbox, tags=tags)
            if f.empty: continue
            f = f.to_crs(crs)
            for g in f.geometry:
                if g is None or g.is_empty: continue
                all_geoms.append(g.buffer(30) if 'Line' in g.geom_type else g)
            print(f"     OSM {list(tags.keys())[0]}: {len(f)}")
        except Exception as e: print(f"     {tags}: {e}")

    # E18 через КЗС
    try:
        r = ox.features_from_bbox(bbox=(29.60,59.96,29.82,60.03),
                                   tags={'highway':['motorway','trunk'],'bridge':True})
        if not r.empty:
            r = r.to_crs(crs)
            for g in r.geometry:
                if g and not g.is_empty: all_geoms.append(g.buffer(40))
            print(f"     E18 bridge: {len(r)}")
    except Exception as e: print(f"     E18: {e}")

    # Жёсткие полигоны
    from shapely.geometry import Polygon as ShPoly
    for coords in DAM_EXCLUSION_POLYS_WGS84:
        p = gpd.GeoDataFrame({'geometry':[ShPoly(coords)]},crs="EPSG:4326").to_crs(crs)
        all_geoms.append(p.geometry.iloc[0])

    if not all_geoms:
        mask = np.zeros(shape, dtype=np.uint8)
    else:
        mask = rio_rasterize([(g,1) for g in all_geoms],
            out_shape=shape, transform=tf, fill=0, dtype='uint8')
        print(f"   Dam mask: {int(np.sum(mask)):,} px")

    prof.update(dtype='uint8', count=1, nodata=255)
    with rasterio.open(out_path,'w',**prof) as dst: dst.write(mask,1)
    del mask; gc.collect()
    return out_path


def apply_dam_exclusion(flood_mask_path, dam_mask_path):
    """Обнуляет пиксели дамб в маске затопления (in-place)."""
    if not dam_mask_path or not os.path.exists(dam_mask_path): return
    with rasterio.open(dam_mask_path) as ds: dam = ds.read(1)
    src = gdal.Open(flood_mask_path, gdal.GA_Update)
    band = src.GetRasterBand(1)
    tile_sz = 4096; rows = src.RasterYSize; cols = src.RasterXSize; n_cl = 0
    for ty in range((rows+tile_sz-1)//tile_sz):
        yo=ty*tile_sz; ys=min(tile_sz,rows-yo)
        for tx in range((cols+tile_sz-1)//tile_sz):
            xo=tx*tile_sz; xs=min(tile_sz,cols-xo)
            m = band.ReadAsArray(xo,yo,xs,ys)
            d = dam[yo:yo+ys,xo:xo+xs]
            if d.shape != m.shape:
                from scipy.ndimage import zoom as ndz
                d = ndz(d,(m.shape[0]/d.shape[0],m.shape[1]/d.shape[1]),order=0).astype(np.uint8)
                d = d[:m.shape[0],:m.shape[1]]
            n_cl += int(np.sum((m==1)&(d==1)))
            m[d==1] = 0; band.WriteArray(m,xo,yo)
    src.FlushCache(); src=None
    print(f"   Dam exclusion: {n_cl:,} px убрано")


def compute_flood_zone_dtm(dtm_path, level_bs77, offset, crs,
                            cov_mask_path=None, dam_mask_path=None,
                            smooth_r=20, simplify_m=20, min_area_m2=2000):
    """Быстрая зона затопления для слайдера (DTM, без connectivity)."""
    thr = level_bs77 + offset
    with rasterio.open(dtm_path) as src:
        dtm = src.read(1).astype(np.float32)
        tf  = src.transform; nodata = src.nodata
    if nodata is not None: dtm[dtm==nodata] = np.nan
    mask = (~np.isnan(dtm) & (dtm<=thr)).astype(np.uint8)

    if cov_mask_path and os.path.exists(cov_mask_path):
        with rasterio.open(cov_mask_path) as cs: cov=cs.read(1)
        if cov.shape!=mask.shape:
            from scipy.ndimage import zoom as ndz
            cov=ndz(cov,(mask.shape[0]/cov.shape[0],mask.shape[1]/cov.shape[1]),order=0)
            cov=cov[:mask.shape[0],:mask.shape[1]].astype(np.uint8)
        mask &= cov

    if dam_mask_path and os.path.exists(dam_mask_path):
        with rasterio.open(dam_mask_path) as ds: dam=ds.read(1)
        if dam.shape!=mask.shape:
            from scipy.ndimage import zoom as ndz
            dam=ndz(dam,(mask.shape[0]/dam.shape[0],mask.shape[1]/dam.shape[1]),order=0)
            dam=dam[:mask.shape[0],:mask.shape[1]].astype(np.uint8)
        mask[dam==1] = 0

    mask = ndimage.binary_closing(mask,ndimage.generate_binary_structure(2,2),iterations=2).astype(np.uint8)
    geoms = [s for s,v in shapes(mask, mask=mask==1, transform=tf)]
    if not geoms: return None, 0

    gdf = gpd.GeoDataFrame({'geometry':geoms}, crs=crs)
    gdf = gdf[gdf.geometry.area>=min_area_m2].copy()
    if gdf.empty: return None, 0
    gdf = gdf.dissolve()
    gdf.geometry = gdf.geometry.buffer(smooth_r).buffer(-smooth_r)
    gdf.geometry = gdf.geometry.simplify(simplify_m)
    gdf = gdf[~gdf.geometry.is_empty]
    ha = gdf.geometry.area.sum()/10000
    gdf_wgs = gdf.to_crs("EPSG:4326")
    gdf_wgs['level'] = level_bs77; gdf_wgs['ha'] = round(ha,0)
    return json.loads(gdf_wgs.to_json()), round(ha, 0)


def build_multilevel_map(dtm_path, final_mask_2m, calib, shift, crs, out_html,
                          dam_mask_path=None, cov_mask_path=None):
    """
    Folium-карта с JS-слайдером уровня затопления 1–5 м.
    Основная зона (FLOOD_LEVEL): полный pipeline (connectivity+SMRF).
    Слайдер: DTM-threshold, быстро, ориентировочно.
    """
    print(f"   Вычисляю зоны для {len(FLOOD_LEVELS_SLIDER)} уровней...")
    offset = calib['offset']

    # Основная зона
    with rasterio.open(final_mask_2m) as src:
        arr=src.read(1); tf_m=src.transform; bounds=src.bounds
    print(f"   Основная зона: {int(np.sum(arr)):,} px")
    geoms_m = [s for s,v in shapes(arr, mask=arr==1, transform=tf_m)]
    del arr; gc.collect()
    main_gdf = gpd.GeoDataFrame({'geometry':geoms_m}, crs=crs)
    main_gdf = main_gdf[main_gdf.geometry.area>=MIN_AREA].dissolve()
    main_gdf.geometry = main_gdf.geometry.buffer(15).buffer(-15)
    main_gdf.geometry = main_gdf.geometry.simplify(SIMPLIFY)
    main_gdf = main_gdf[~main_gdf.geometry.is_empty]
    main_ha = main_gdf.geometry.area.sum()/10000
    main_wgs = main_gdf.to_crs("EPSG:4326")
    main_gj = json.loads(main_wgs.to_json())
    print(f"   Основная зона: {main_ha:.0f} га")

    # Landuse
    lu_html = ""
    try:
        import osmnx as ox; ox.settings.timeout=60
        fb = main_wgs.total_bounds
        lu = ox.features_from_bbox(bbox=(fb[0],fb[1],fb[2],fb[3]),tags={'landuse':True})
        if not lu.empty:
            lu = lu[lu.geometry.type.isin(['Polygon','MultiPolygon'])].to_crs(crs)
            imp = gpd.overlay(lu, main_gdf, how='intersection')
            imp['ha'] = imp.geometry.area/10000
            top = imp.groupby('landuse')['ha'].sum().sort_values(ascending=False).head(8)
            print("\n   Landuse:"); print(top.to_string())
            rows_lu = "".join(f"<tr><td>{k}</td><td>{v:.0f}</td></tr>" for k,v in top.items())
            lu_html = (f'<table style="font-size:11px;border-collapse:collapse;margin-top:4px">'
                       f'<tr><th style="text-align:left">Тип</th><th>га</th></tr>'
                       f'{rows_lu}</table>')
    except Exception as e: print(f"   landuse: {e}")

    # Зоны слайдера
    slider_data = {}
    for lvl in FLOOD_LEVELS_SLIDER:
        gj, ha_lvl = compute_flood_zone_dtm(
            dtm_path, lvl, offset, crs,
            cov_mask_path=cov_mask_path, dam_mask_path=dam_mask_path)
        k = f"{lvl:.1f}"
        slider_data[k] = {'geojson': gj or {"type":"FeatureCollection","features":[]}, 'ha': ha_lvl}
        print(f"     {k} м: {ha_lvl:.0f} га")

    # Folium
    bw = gpd.GeoDataFrame({'geometry':[box(bounds.left,bounds.bottom,bounds.right,bounds.top)]},
                           crs=crs).to_crs("EPSG:4326").total_bounds
    cy,cx = (bw[1]+bw[3])/2,(bw[0]+bw[2])/2
    m = folium.Map(location=[cy,cx], zoom_start=11, tiles='CartoDB positron')

    # GPS КЗС маркеры
    for name,lat,lon,h,_ in GPS_KZS_POINTS:
        folium.CircleMarker([lat,lon],radius=7,color='#FF6600',fill=True,
            fill_color='#FF6600',fill_opacity=0.9,
            tooltip=f"{name}: {h:.3f} м БС-77").add_to(m)

    # Метапанель
    rmse_b = shift.get('rmse_before', float('nan'))
    rmse_a = shift.get('rmse_after', float('nan'))
    rmse_s = f"{rmse_b:.3f}→{rmse_a:.3f}" if not (isinstance(rmse_b,float) and rmse_b!=rmse_b) else "N/A"
    info = (f'<div id="info-panel" style="position:fixed;top:10px;left:10px;z-index:900;'
            f'background:white;padding:14px 16px;border-radius:10px;'
            f'box-shadow:0 2px 12px rgba(0,0,0,0.25);font-size:12px;line-height:1.7em;'
            f'min-width:210px;max-width:260px">'
            f'<b style="font-size:13px">🌊 Flood Map v9</b><br>'
            f'<span style="color:#555">Дата: {STRIP_DATE}</span><br>'
            f'Offset: <b>{calib["offset"]:.3f}</b> м [{calib["note"]}]<br>'
            f'CI 95%: [{calib["ci_lo"]:.3f}, {calib["ci_hi"]:.3f}] м<br>'
            f'Coreg dz: {shift["dz"]:+.3f} м (RMSE {rmse_s})<br>'
            f'SMRF w={SMRF_WINDOW_MAX} м + КЗС маска<br>'
            f'<hr style="margin:5px 0">'
            f'<span style="color:#0066cc">■</span> '
            f'<b>Точная зона {FLOOD_LEVEL} м:</b> {main_ha:.0f} га<br>'
            f'{lu_html}</div>')
    m.get_root().html.add_child(folium.Element(info))

    # Строим JS с данными
    lvls_js = "[" + ",".join(f'"{k}"' for k in slider_data) + "]"
    ha_js   = "{" + ",".join(f'"{k}":{v["ha"]}' for k,v in slider_data.items()) + "}"
    zones_parts = [f'  "{k}": {json.dumps(v["geojson"])}' for k,v in slider_data.items()]
    zones_js = "{\n" + ",\n".join(zones_parts) + "\n}"
    main_lvl_idx = next((i for i,k in enumerate(slider_data) if abs(float(k)-FLOOD_LEVEL)<0.01), 4)

    js = f"""
<script>
var _zones = {zones_js};
var _levels = {lvls_js};
var _ha = {ha_js};
var _mainGJ = {json.dumps(main_gj)};
var _mainL = null;
var _sliderLayers = {{}};
var _currentKey = null;

var COLORS = {{
  "1.0":"#b8d4ff","1.5":"#80aeff","2.0":"#4d88ff","2.5":"#1a5fff",
  "3.0":"#0040dd","3.5":"#e06000","4.0":"#c04000","4.5":"#a02000","5.0":"#800000"
}};

function getLeafletMap() {{
  var els = document.querySelectorAll(".leaflet-container");
  for(var i=0;i<els.length;i++) {{if(els[i]._leaflet_map) return els[i]._leaflet_map;}}
  for(var k in window) {{try{{if(window[k]&&window[k].addLayer&&window[k].eachLayer) return window[k];}}catch(e){{}}}}
  return null;
}}

function switchLevel(lvlKey) {{
  if(lvlKey === _currentKey) return;
  var mp = getLeafletMap(); if(!mp) return;
  if(_currentKey && _sliderLayers[_currentKey]) {{
    mp.removeLayer(_sliderLayers[_currentKey]);
  }}
  _currentKey = lvlKey;
  if(_sliderLayers[lvlKey]) {{ _sliderLayers[lvlKey].addTo(mp); }}
  document.getElementById(\"lvl-val\").textContent = lvlKey + \" м БС-77\";
  document.getElementById(\"ha-val\").textContent  = \"~\" + _ha[lvlKey] + \" га (DTM)\";
}}


document.addEventListener("DOMContentLoaded",function(){{
  setTimeout(function(){{
    var mp = getLeafletMap(); if(!mp) return;
    if(_mainGJ) {{
      _mainL = L.geoJSON(_mainGJ, {{style:{{fillColor:\"#1E90FF\",color:\"#0000CD\",weight:1,fillOpacity:0.65}}}});
      _mainL.addTo(mp);
    }}
    // Pre-create all slider layers (hidden) for instant switching
    for(var _i = 0; _i < _levels.length; _i++) {{
      (function(k){{
        var gj = _zones[k];
        if(gj && gj.features && gj.features.length > 0) {{
          _sliderLayers[k] = L.geoJSON(gj, {{
            style: {{fillColor:COLORS[k]||\"#4d88ff\",color:\"#003\",weight:0.7,fillOpacity:0.45}},
            onEachFeature: function(f,l){{ l.bindTooltip(k+\" м: ~\"+_ha[k]+\" га\",{{sticky:true}}); }}
          }});
        }}
      }})(_levels[_i]);
    }}
    var panel = document.createElement("div");
    panel.style.cssText = "position:fixed;bottom:18px;left:50%;transform:translateX(-50%);z-index:1002;background:white;padding:13px 22px;border-radius:12px;box-shadow:0 2px 14px rgba(0,0,0,0.28);font-family:sans-serif;font-size:13px;text-align:center;min-width:360px";
    panel.innerHTML = (
      '<div style="margin-bottom:7px">' +
      '<b>🌊 Уровень наводнения:</b> ' +
      '<span id="lvl-val" style="color:#cc2000;font-size:16px;font-weight:bold;margin-left:6px">–</span>' +
      '<span id="ha-val" style="color:#666;font-size:11px;margin-left:10px"></span></div>' +
      '<input type="range" id="fslider" min="0" max="' + (_levels.length-1) + '" step="1" value="{main_lvl_idx}" ' +
      'style="width:300px;cursor:pointer;accent-color:#0050cc"><br>' +
      '<div style="display:flex;justify-content:space-between;width:300px;margin:2px auto 0;font-size:10px;color:#888">' +
      '<span>1.0</span><span>2.0</span><span>3.0</span><span>4.0</span><span>5.0</span></div>' +
      '<div style="margin-top:8px;font-size:10px;color:#777">' +
      '<span style="background:#1E90FF;padding:1px 8px;color:white;border-radius:3px">■</span> ' +
      'Точный уровень {FLOOD_LEVEL} м &nbsp;&nbsp;' +
      '<span style="background:#e06000;padding:1px 8px;color:white;border-radius:3px">■</span> ' +
      'Слайдер (DTM)</div>'
    );
    document.body.appendChild(panel);

    var sl = document.getElementById("fslider");
    sl.addEventListener("input",function(){{ switchLevel(_levels[+this.value]); }});
    switchLevel(_levels[+sl.value]);
  }}, 900);
}});
</script>
""".replace("{main_lvl_idx}", str(main_lvl_idx)).replace("{FLOOD_LEVEL}", str(FLOOD_LEVEL))

    m.get_root().html.add_child(folium.Element(js))
    # DEM topo heatmap layer
    try:
        import matplotlib; matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import matplotlib.colors as mcolors
        import base64
        from PIL import Image as PILImage

        dem_png = os.path.join(WD, 'dem_heatmap.png')
        with rasterio.open(dtm_path) as ds:
            scale = 6
            dtm_sm = ds.read(1,
                out_shape=(ds.height//scale, ds.width//scale),
                resampling=rasterio.enums.Resampling.average).astype(np.float32)
            raw_bounds = ds.bounds; raw_crs = ds.crs
        dtm_sm[dtm_sm == -9999] = np.nan

        bnd4 = gpd.GeoDataFrame(
            {'geometry':[box(raw_bounds.left,raw_bounds.bottom,
                             raw_bounds.right,raw_bounds.top)]},
            crs=raw_crs).to_crs('EPSG:4326').total_bounds
        img_bounds = [[float(bnd4[1]),float(bnd4[0])],[float(bnd4[3]),float(bnd4[2])]]

        norm = mcolors.Normalize(vmin=-2, vmax=60)
        cmap = plt.get_cmap('terrain')
        rgba = (cmap(norm(np.where(np.isnan(dtm_sm), -10, dtm_sm))) * 255).astype(np.uint8)
        rgba[np.isnan(dtm_sm)] = [50, 100, 180, 0]
        pil = PILImage.fromarray(rgba, 'RGBA').transpose(PILImage.FLIP_TOP_BOTTOM)
        pil.save(dem_png)

        with open(dem_png,'rb') as _f: dem_b64 = base64.b64encode(_f.read()).decode()
        folium.raster_layers.ImageOverlay(
            image='data:image/png;base64,'+dem_b64,
            bounds=img_bounds, opacity=0.72,
            name='Рельеф DTM (тепловая карта)',
            show=False, interactive=False,
        ).add_to(m)

        # Colorbar as fixed div
        fig_cb, ax_cb = plt.subplots(figsize=(3.8, 0.35))
        fig_cb.subplots_adjust(bottom=0.55)
        matplotlib.colorbar.ColorbarBase(ax_cb, cmap=cmap, norm=norm, orientation='horizontal')
        ax_cb.set_xlabel('Высота DTM, м (прибл.)', fontsize=7)
        cb_path = os.path.join(WD,'dem_colorbar.png')
        fig_cb.savefig(cb_path,dpi=72,bbox_inches='tight',transparent=True); plt.close(fig_cb)
        with open(cb_path,'rb') as _f: cb_b64 = base64.b64encode(_f.read()).decode()

        # Build elevation grid for click queries
        step = max(1, dtm_sm.shape[0]//180)
        grid = dtm_sm[::step,::step]
        gr, gc = grid.shape
        gflat = [round(float(v),1) if not np.isnan(v) else None for v in grid.flatten()]

        dem_ui = (
            '<div id="dem-cb" style="display:none;position:fixed;bottom:98px;left:10px;'
            'z-index:950;background:rgba(255,255,255,0.93);padding:7px 10px;'
            'border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,0.18);font-size:10.5px">'
            '<img src="data:image/png;base64,'+cb_b64+'" style="width:250px"><br>'
            '<span id="dem-info">Включите слой и кликните на карту</span></div>'
        )
        m.get_root().html.add_child(folium.Element(dem_ui))
        print("   DEM heatmap layer: OK")
    except Exception as _dem_err:
        print(f"   DEM heatmap skip: {_dem_err}")

    folium.LayerControl().add_to(m)
    m.save(out_html)
    print(f"   Интерактивная карта: {out_html}")
    return m


print("[MODULES 1-7 + v9] Все функции загружены")


In [ ]:
# ======================================================================================
# MAIN — FLOOD MAP v9
# ======================================================================================

def main():
    print("="*70)
    print(" FLOOD MAP v9 — SMRF + ICESat-2 + GPS КЗС + дамбы + слайдер 1-5м")
    print(f" Уровень: {FLOOD_LEVEL} м БС-77  |  Дата: {STRIP_DATE}")
    print(f" EGM2008={EGM_A0:.3f} м | δ_BS77={DELTA_BS77:+.2f} м")
    print(f" SMRF: slope={SMRF_SLOPE}, window={SMRF_WINDOW_MAX} м")
    print("="*70)

    dsm_coreg   = os.path.join(WD, "dsm_coregistered.tif")
    stable_mask = os.path.join(WD, "stable_surface_mask.tif")
    cov_mask    = os.path.join(WD, "coverage_mask.tif")
    water_mask  = os.path.join(WD, "osm_water_mask.tif")
    dam_mask    = os.path.join(WD, "dam_mask_10m.tif")
    dtm_10m     = os.path.join(WD, "dtm_smrf_10m.tif")
    mask_2m     = os.path.join(WD, "mask_2m.tif")
    conn_10m    = os.path.join(WD, "conn_10m.tif")
    final_2m    = os.path.join(WD, "flood_final_2m.tif")
    prob_map    = os.path.join(WD, "flood_probability.tif")
    out_html    = os.path.join(WD, "spb_flood_v9_interactive.html")
    meta_json   = os.path.join(WD, "calibration_v9.json")

    # 0. ICESat-2
    print("\n[0/8] ICESat-2 ATL08 → Nuth & Kääb...")
    shift = {'dx':0.,'dy':0.,'dz':0.,'rmse_before':float('nan'),'rmse_after':float('nan'),'n':0}
    atl_files = download_atl08(date_end=STRIP_DATE)
    if atl_files:
        build_stable_mask(INPUT_DSM, CRS_METRIC, stable_mask)
        parts = []
        for f in atl_files:
            g = parse_atl08(f, CRS_METRIC)
            if not g.empty: parts.append(g); print(f"   {os.path.basename(f)}: {len(g)} точек")
        if parts:
            atl_gdf = gpd.GeoDataFrame(pd.concat(parts,ignore_index=True),crs=CRS_METRIC)
            print(f"   ATL08 итого: {len(atl_gdf):,} точек")
            shift = nuth_kaab(INPUT_DSM, atl_gdf, stable_mask, CRS_METRIC)
        else: print("   ATL08: нет точек")
    dsm_work = apply_shift(INPUT_DSM, shift, dsm_coreg)
    print(f"   Сдвиг: dx={shift["dx"]:+.3f} dy={shift["dy"]:+.3f} dz={shift["dz"]:+.3f} м")
    gc.collect()

    # 1. Калибровка (GPS + дороги + посты)
    print("\n[1/8] Калибровка (GPS КЗС + дороги + водомерные посты)...")
    road_pts = extract_road_elevations(dsm_work, CRS_METRIC, max_pts=200)
    calib = calibrate(dsm_work, CONTROL_POINTS, GAUGE_CSV, CRS_METRIC, road_elev_pts=road_pts)
    offset    = calib['offset']
    threshold = FLOOD_LEVEL + offset
    wl_level  = offset
    gauge_wl  = get_gauge_levels(GAUGE_CSV, STRIP_DATE)
    print(f"\n   offset={offset:.4f} м | threshold={threshold:.4f} м")
    print(f"   CI=[{calib['ci_lo']:.3f},{calib['ci_hi']:.3f}] м")
    gc.collect()

    # 2. Маска покрытия
    print("\n[2/8] Маска покрытия...")
    create_coverage_mask(dsm_work, cov_mask, CRS_METRIC)
    gc.collect()

    # 3. Гидрокоррекция
    print("\n[3/8] Гидрокоррекция...")
    print("   [3a] OSM water mask...")
    build_osm_water_mask(dsm_work, water_mask, CRS_METRIC)
    gc.collect()
    print("   [3b] Water flattening...")
    water_flattening(CRS_METRIC, offset, gauge_wl)
    gc.collect()
    print("   [3c] SMRF: DSM → DTM...")
    apply_smrf(dsm_work, dtm_10m, CRS_METRIC,
               slope_thr=SMRF_SLOPE, window_max_m=SMRF_WINDOW_MAX,
               cell_m=SMRF_CELL, cov_mask_path=cov_mask,
               csf_mult=calib.get('csf_mult',1.0))
    gc.collect()

    # 4. Маска дамб КЗС
    print("\n[4/8] Маска КЗС и дамб...")
    build_dam_mask(CRS_METRIC, dam_mask)
    gc.collect()

    # 5. Пороговая маска 2 м
    print(f"\n[5/8] Пороговая маска 2 м (threshold={threshold:.3f} м)...")
    threshold_mask_tiled(dsm_work, mask_2m, threshold, wl_level,
                         water_mask_path=water_mask)
    apply_dam_exclusion(mask_2m, dam_mask)
    gc.collect()

    # 6. Связность + DTM
    print("\n[6/8] Связность + DTM-коррекция...")
    connectivity_analysis(mask_2m, conn_10m, WATER_SEEDS, CRS_METRIC,
                          CLOSING_M, CONN_RES, SEED_R,
                          cov_path=cov_mask, dtm_path=dtm_10m, threshold=threshold)
    gc.collect()

    # 7. Финальная маска + вероятность
    print("\n[7/8] Финальная маска + вероятность...")
    apply_filter_2m(mask_2m, conn_10m, final_2m)
    apply_dam_exclusion(final_2m, dam_mask)
    build_probability_map(dsm_work, prob_map, calib, FLOOD_LEVEL, n_samples=50)
    for f in [mask_2m, conn_10m]:
        if os.path.exists(f): os.remove(f)
    gc.collect()

    # 8. Интерактивная карта со слайдером
    print("\n[8/8] Интерактивная карта со слайдером 1–5 м...")
    build_multilevel_map(
        dtm_path=dtm_10m, final_mask_2m=final_2m,
        calib=calib, shift=shift, crs=CRS_METRIC, out_html=out_html,
        dam_mask_path=dam_mask, cov_mask_path=cov_mask,
    )

    meta = {
        "version": "v9",
        "strip_date": STRIP_DATE,
        "flood_level_bs77": FLOOD_LEVEL,
        "slider_levels": FLOOD_LEVELS_SLIDER,
        "calibration": {"offset": offset, "ci_lo": calib['ci_lo'],
                        "ci_hi": calib['ci_hi'], "source": calib['note'],
                        "road_pts_used": len(road_pts),
                        "gps_kzs_pts": len(GPS_KZS_POINTS)},
        "coregistration": {"dx": shift['dx'], "dy": shift['dy'], "dz": shift['dz'],
                           "rmse_before": shift['rmse_before'], "rmse_after": shift['rmse_after']},
        "dam_exclusion_polys": len(DAM_EXCLUSION_POLYS_WGS84),
        "smrf": {"slope": SMRF_SLOPE, "window_max_m": SMRF_WINDOW_MAX},
        "threshold_dsm": threshold,
    }
    with open(meta_json,'w',encoding='utf-8') as jf: json.dump(meta,jf,ensure_ascii=False,indent=2)

    print(f"\n{'='*70}")
    print(f" ГОТОВО! Flood Map v9")
    print(f" Интерактивная карта: {out_html}")
    print(f" Маска 2 м:           {final_2m}")
    print(f" Вероятность:         {prob_map}")
    print(f" Метаданные:          {meta_json}")
    print(f"{'='*70}")


main()


In [ ]:
# ======================================================================================
# КРОСС-СВЕРКА — запускается независимо
# ======================================================================================
import numpy as np, geopandas as gpd, pandas as pd, rasterio, os
from shapely.geometry import Point
from osgeo import gdal

dem_path = os.path.join(WD, "dsm_coregistered.tif")
if not os.path.exists(dem_path):
    dem_path = INPUT_DSM
    print(f"⚠ Используем исходный DSM: {dem_path}")

# 10м для скорости
dem10 = os.path.join(WD, "dem_10m_check.tif")
if not os.path.exists(dem10):
    gdal.Warp(dem10, dem_path, options=gdal.WarpOptions(
        format="GTiff",xRes=10,yRes=10,resampleAlg="bilinear",dstNodata=-9999,
        creationOptions=["COMPRESS=LZW","BIGTIFF=YES"]))

with rasterio.open(dem10) as s:
    dem_arr=s.read(1).astype(float); nd=s.nodata; tf=s.transform
    if nd is not None: dem_arr[dem_arr==nd]=np.nan

# GEOID_OFFSET из JSON если есть
meta_f = os.path.join(WD, "calibration_v8.json")
if os.path.exists(meta_f):
    import json
    with open(meta_f) as jf: meta=json.load(jf)
    GEOID_OFFSET = meta['calibration']['offset']
    print(f"Offset из метаданных: {GEOID_OFFSET:.4f} м")
else:
    GEOID_OFFSET = SPB_OFFSET_FALLBACK
    print(f"Offset fallback: {GEOID_OFFSET:.4f} м")

def samp(arr,tf,lon,lat,crs,r=5):
    pt=gpd.GeoDataFrame({'geometry':[Point(lon,lat)]},crs="EPSG:4326").to_crs(crs)
    px,py=pt.geometry.iloc[0].x,pt.geometry.iloc[0].y
    c=int((px-tf.c)/tf.a); row=int((py-tf.f)/tf.e)
    r0,r1=max(0,row-r),min(arr.shape[0],row+r+1)
    c0,c1=max(0,c-r),min(arr.shape[1],c+r+1)
    v=arr[r0:r1,c0:c1]; v=v[~np.isnan(v)&(v>-5)&(v<300)]
    return float(np.median(v)) if len(v) else np.nan

print(f"\n{'Точка':<38} {'H_ort':>8} {'Ожид.':>7} {'Δ':>7} {'Источник'}")
print("-"*75)
errors=[]
for name,lat,lon,H,src_label in HARD_POINTS:
    v=samp(dem_arr,tf,lon,lat,CRS_METRIC)
    if np.isnan(v): print(f"{name:<38} {'N/A':>8} {H:>7.2f}       {src_label}"); continue
    ort=v-GEOID_OFFSET; dlt=ort-H; errors.append(dlt)
    ok="✓" if abs(dlt)<TOLERANCE_GOOD else ("≈" if abs(dlt)<TOLERANCE_OK else "⚠")
    diag=" (крыша?)" if dlt>10 else (" (вода?)" if dlt<-5 else "")
    print(f"{name:<38} {ort:>8.2f} {H:>7.2f} {dlt:>+7.2f} {src_label} {ok}{diag}")

if errors:
    e=np.array(errors)
    rms=np.sqrt(np.mean(e**2))
    print(f"\n  Среднее: {np.mean(e):+.3f} м | Стд: {np.std(e):.3f} м | RMS: {rms:.3f} м")
    q = ("ОТЛИЧНО" if rms<TOLERANCE_GOOD else
         "ХОРОШО (DSM-эффект)" if rms<TOLERANCE_OK else
         "ПРИЕМЛЕМО" if rms<10 else "⚠ ПЛОХО")
    print(f"  Качество: {q}")
